# Short and Still Episodes in MolmoAct2

In this notebook, we demonstrate filtering out episodes that are too short (under 3 s) or motionless, using queries with the [DataFusion DataFrame API](https://datafusion.apache.org/python/).

## Prerequisites

If you have not done so yet, prepare MolmoAct2 RRD files to play with by either (1) downloading the source data and converting it to RRD, or (2) downloading the converted RRD files directly.

Then register them in the local catalog on your machine (see the README's [Local Catalog](../README.md#4-local-catalog) section):

```bash
pixi run serve              # in one shell, leave running
pixi run -e molmo register  # in another shell
```

In [ ]:
from __future__ import annotations

import rerun as rr
from datafusion import Expr, col, lit
from datafusion import functions as F
from rerun.notebook import Viewer
from rerun.utilities.datafusion.functions.url_generation import segment_url

SERVER = "rerun+http://127.0.0.1:51234"

In [ ]:
client = rr.catalog.CatalogClient(SERVER)

dataset_name = "molmoact2"
dataset = client.get_dataset(dataset_name)

## Shared setup

In [ ]:
FPS = 30

# Two thresholds decide which episodes are dropped:
# - `MIN_SECONDS`: an episode shorter than this many seconds is short.
# - `MIN_JOINT_SPREAD`: an arm moves when its joint spread is above this many radians, and an episode is still when neither arm moves.
MIN_SECONDS = 3.0
MIN_JOINT_SPREAD = 0.1


# Segment properties show up as length-1 list columns named `property:<group>:<key>`, so `prop()` extracts the scalar.
def prop(name: str) -> Expr:
    """Segment-property columns are length-1 lists; take the scalar."""
    return col(f'"{name}"')[0]


segments = dataset.segment_table()
# The joint spread is the largest `q99 − q01` of the arm's six commanded joints (see the README's [Stats layer](../README.md#stats-layer) section).
if "property:stats:left_joint_spread" not in segments.schema().names:
    raise RuntimeError(
        "No episode has the stats layer. Build it with `pixi run -e molmo convert-stats`, then register."
    )

print("Total segments:", segments.count())

## Query 1: Short and still episodes

Give each episode two flags, `short` and `still`.
`KEEP` marks the episodes to keep, which are neither short nor still.

In [ ]:
episodes = (
    segments.select(
        "rerun_segment_id",
        prop("property:episode:episode_index").alias("episode"),
        prop("property:episode:task").alias("task"),
        (prop("property:episode:length") / lit(float(FPS))).alias("duration_s"),
        prop("property:stats:left_joint_spread").alias("left_spread"),
        prop("property:stats:right_joint_spread").alias("right_spread"),
    )
    .with_column("short", col("duration_s") < lit(MIN_SECONDS))
    .with_column(
        "still",
        # Without spreads the comparison is null, which `coalesce` turns into false.
        F.coalesce(
            (col("left_spread") < lit(MIN_JOINT_SPREAD)) & (col("right_spread") < lit(MIN_JOINT_SPREAD)), lit(False)
        ),
    )
)
KEEP = ~col("short") & ~col("still")

## Query 2: The dropped episodes

The viewer below lists the **dropped** episodes, shortest first, with their joint spreads in radians.

Click an episode's `url` to open it in the same viewer, and pick the `Dropped episodes` table in the viewer's left panel to go back to the list.

In [ ]:
dropped = (
    episodes.filter(~KEEP)
    .with_column("url", segment_url(dataset))
    .select(
        "url",
        "episode",
        "task",
        F.round(col("duration_s"), lit(1)).alias("duration_s"),
        F.round(col("left_spread"), lit(4)).alias("left_spread"),
        F.round(col("right_spread"), lit(4)).alias("right_spread"),
        "short",
        "still",
    )
    .sort(col("duration_s").sort())
)

print("Dropped episodes:", dropped.count())

viewer = Viewer(width=1200, height=820)
viewer.display()
viewer.send_table("Dropped episodes", dropped)

## Query 3: The kept episodes

The viewer below lists the **kept** episodes.

Click an episode's `url` to open it in the same viewer, and pick the `Kept episodes` table in the viewer's left panel to go back to the list.

In [ ]:
kept = dataset.filter_segments(episodes.filter(KEEP))
print(f"{len(kept.segment_ids())} of {len(dataset.segment_ids())} episode(s) kept")

kept_episodes = (
    kept.segment_table()
    .with_column("url", segment_url(dataset))
    .select(
        "url",
        prop("property:episode:episode_index").alias("episode"),
        prop("property:episode:task").alias("task"),
        prop("property:episode:length").alias("length"),
    )
    .sort(col("episode").sort())
)

print("Kept episodes:", kept_episodes.count())

viewer = Viewer(width=1200, height=820)
viewer.display()
viewer.send_table("Kept episodes", kept_episodes)